In [1]:
import json
with open("/content/Students.JSON") as f:
  students = json.load(f)
print(students)

[{'id': 1, 'name': 'Maria Santos', 'section': '7-Ampere', 'subjects': {'Math2': 1.5, 'Math3': 1.75, 'Biology1': 2.0, 'Chem1': 1.25, 'Physics1': 1.75, 'Computer Science2': 1.25, 'PEHM2': 1.0, 'VE2': 1.75, 'Social Science2': 1.5, 'English2': 1.25, 'Filipino2': 1.75}}, {'id': 2, 'name': 'Juan Dela Cruz', 'section': '7-Curie', 'subjects': {'Math2': 2.5, 'Math3': 2.75, 'Biology1': 2.0, 'Chem1': 2.25, 'Physics1': 2.5, 'Computer Science2': 2.0, 'PEHM2': 1.75, 'VE2': 2.5, 'Social Science2': 2.25, 'English2': 2.0, 'Filipino2': 2.75}}, {'id': 3, 'name': 'Ana Reyes', 'section': '7-Rutherford', 'subjects': {'Math2': 1.0, 'Math3': 1.25, 'Biology1': 1.5, 'Chem1': 1.0, 'Physics1': 1.25, 'Computer Science2': 1.0, 'PEHM2': 1.25, 'VE2': 1.25, 'Social Science2': 1.0, 'English2': 1.25, 'Filipino2': 1.5}}, {'id': 4, 'name': 'Carlo Mendoza', 'section': '7-Newton', 'subjects': {'Math2': 2.75, 'Math3': 3.0, 'Biology1': 2.5, 'Chem1': 2.75, 'Physics1': 2.5, 'Computer Science2': 2.25, 'PEHM2': 2.0, 'VE2': 2.5, '

In [2]:
!pip install firebase-admin

In [3]:
import firebase_admin
from firebase_admin import credentials

path_to_key = "/content/Firebase.JSON"

if not firebase_admin._apps:
    cred = credentials.Certificate(path_to_key)
    firebase_admin.initialize_app(cred, {
        'databaseURL': "https://jasmine-daradar-g8-library-db-default-rtdb.asia-southeast1.firebasedatabase.app/"
    })
    print("Firebase connected successfully!")
else:
    print("Firebase is already connected.")

Firebase connected successfully!


In [4]:
from firebase_admin import db

ref = db.reference("students")

for student in students:
    ref.child(str(student["id"])).set(student)

print("Data uploaded successfully!")

Data uploaded successfully!


In [5]:
student_id_to_average = input("Enter the ID of the student to calculate the general average for: ")

student_ref = db.reference(f"students/{student_id_to_average}")
student_data = student_ref.get()

if student_data:
    subjects = student_data.get("subjects")
    if subjects:
        total_grades = sum(subjects.values())
        num_subjects = len(subjects)
        if num_subjects > 0:
            general_average = total_grades / num_subjects
            print(f"The general average for {student_data['name']} (ID: {student_id_to_average}) is: {general_average:.2f}")
        else:
            print(f"No subjects found for student ID {student_id_to_average}.")
    else:
        print(f"No subjects data found for student ID {student_id_to_average}.")
else:
    print(f"Student with ID {student_id_to_average} not found in the database.")

Enter the ID of the student to calculate the general average for: 2
The general average for Juan Dela Cruz (ID: 2) is: 2.30


In [26]:
exit_overall_program = False
while True:
    print("\n===== STUDENT DATABASE =====")
    print("1. Display Students")
    print("2. Add Student and Info")
    print("3. Update Student Info")
    print("4. Remove Student Record")
    print("5. Features")
    print("6. Exit Program")

    choice = input("Enter choice: ")

   # 1. DISPLAY ALL STUDENTS & GRADES
    if choice == "1":
        students_data = ref.get()
        if students_data:
            # Firebase returns a dictionary, convert to list of students or handle None values
            students_list = [student for student in students_data if student is not None]
            if students_list:
                for student in students_list:
                    print(f"\n--- Record for {student.get('name')} (ID: {student.get('id')}) ---")
                    print(f"Section: {student.get('section')}")

                    subjects = student.get('subjects', {})
                    if subjects:
                        print(f"{'Subject':<25} | {'Grade':<5}")
                        print("-" * 33)
                        for sub_name, grade in subjects.items():
                            print(f"{sub_name:<25} | {grade:<5}")
                    else:
                        print(f"No subjects found for {student.get('name')}.")
            else:
                print("No student records found.")
        else:
            print("No records found.")

 # 2. ADD STUDENT
    elif choice == "2":
        sid = input("Enter Student ID: ")
        name = input("Enter Full Name: ")
        section = input("Enter Section (e.g., 7-Ampere): ")

        print("\n--- Enter Grades for All Subjects ---")
        # Collecting all 11 subjects based on your JSON structure
        m2 = float(input("Math2: "))
        m3 = float(input("Math3: "))
        bio1 = float(input("Biology1: "))
        chem1 = float(input("Chem1: "))
        phys1 = float(input("Physics1: "))
        cs2 = float(input("Computer Science2: "))
        pehm2 = float(input("PEHM2: "))
        ve2 = float(input("VE2: "))
        ss2 = float(input("Social Science2: "))
        eng2 = float(input("English2: "))
        fil2 = float(input("Filipino2: "))

        new_student = {
            "id": int(sid),
            "name": name,
            "section": section,
            "subjects": {
                "Math2": m2,
                "Math3": m3,
                "Biology1": bio1,
                "Chem1": chem1,
                "Physics1": phys1,
                "Computer Science2": cs2,
                "PEHM2": pehm2,
                "VE2": ve2,
                "Social Science2": ss2,
                "English2": eng2,
                "Filipino2": fil2
            }
        }

        # Saving to Firebase using the ID as the key
        ref.child(sid).set(new_student)
        print(f"\nSuccess: {name} has been added!")

    # 3. UPDATE
    elif choice == "3":
        sid = input("Enter ID to update: ")
        student_ref = ref.child(sid)
        if student_ref.get():
            field = input("Which subject to update? (e.g., Math2, Biology1): ")
            new_grade = float(input(f"Enter new grade for {field}: "))

            # Updates just that specific subject grade
            student_ref.update({f"subjects/{field}": new_grade})
            print("Record updated!")
        else:
            print("Student not found.")

    # 4. REMOVE
    elif choice == "4":
        sid = input("Enter ID to remove: ")
        if ref.child(sid).get():
            ref.child(sid).delete()
            print("Record deleted.")
        else:
            print("Record not found.")

    # 5. FEATURES
    elif choice == "5":
      print("FEATURES:")
      while True:
        print("\n===== DATABASE FEATURES ====")
        print("1. Display Average Grade")
        print("2. Display Highest Grade")
        print("3. Display Lowest Grade")
        print("4. Exit Features Menu")
        print("5. Exit Features And main menu")

        feature_choice = input("Enter choice: ")

        # NOTE: Code beyond this point is assisted by Google Gemini.

        # 1A. DISPLAY AVERAGE GRADE
        if feature_choice == "1":
          students_data = ref.get()
          if students_data:
            students_list = [student for student in students_data if student is not None]
            if students_list:
                print("\n--- General Averages for Each Student ---")
                for student in students_list:
                    name = student.get('name')
                    subjects = student.get('subjects', {})
                    if subjects:
                        total_grades = sum(subjects.values())
                        num_subjects = len(subjects)
                        if num_subjects > 0:
                            general_average = total_grades / num_subjects
                            print(f"{name}: {general_average:.2f}")
                        else:
                            print(f"{name}: No subjects found to calculate average.")
                    else:
                        print(f"{name}: No subjects data found.")
            else:
                print("No student records found.")
          else:
            print("No records found.")

        # 2A. HIGHEST GRADE
        elif feature_choice == "2":
          students_data = ref.get()
          if students_data:
            students_list = [student for student in students_data if student is not None]
            if students_list:
                print("\n--- Highest Grades for Each Student ---")
                for student in students_list:
                    name = student.get('name')
                    subjects = student.get('subjects', {})
                    if subjects:
                        # Assuming lower numerical grades are 'higher' or 'better' grades
                        highest_grade_value = min(subjects.values())
                        # Find all subjects that have this 'highest' grade
                        highest_grade_subjects = [sub for sub, grade in subjects.items() if grade == highest_grade_value]
                        print(f"{name}: {', '.join(highest_grade_subjects)} - {highest_grade_value:.2f}")
                    else:
                        print(f"{name}: No subjects found.")
            else:
                print("No student records found.")
          else:
            print("No records found.")

        # 3A. LOWEST GRADE
        elif feature_choice == "3":
          students_data = ref.get()
          if students_data:
            students_list = [student for student in students_data if student is not None]
            if students_list:
                print("\n--- Lowest Grades for Each Student ---")
                for student in students_list:
                    name = student.get('name')
                    subjects = student.get('subjects', {})
                    if subjects:
                        # Assuming higher numerical grades are 'lower' or 'worse' grades
                        lowest_grade_value = max(subjects.values())
                        # Find all subjects that have this 'lowest' grade
                        lowest_grade_subjects = [sub for sub, grade in subjects.items() if grade == lowest_grade_value]
                        print(f"{name}: {', '.join(lowest_grade_subjects)} - {lowest_grade_value:.2f}")
                    else:
                        print(f"{name}: No subjects found.")
            else:
                print("No student records found.")
          else:
            print("No records found.")

        # 4A. Exit Features Menu
        elif feature_choice == "4":
          print("Exiting Features Menu...")
          break

        # 5A. Just quit off
        elif feature_choice == "5":
            print("Exiting Program...")
            exit_overall_program = True
            break

        else:
            print("Invalid choice. Please try again.")

      if exit_overall_program:
          break


    # 6. Exit Program
    elif choice == "6":
        print("Exiting Program...")
        break

    else:
        print("Invalid choice. Please try again.")


===== STUDENT DATABASE =====
1. Display Students
2. Add Student and Info
3. Update Student Info
4. Remove Student Record
5. Features
6. Exit Program
Enter choice: 5
FEATURES:

===== DATABASE FEATURES ====
1. Display Average Grade
2. Display Highest Grade
3. Display Lowest Grade
4. Exit Features Menu
5. Exit Features And main menu
Enter choice: 1

--- General Averages for Each Student ---
Maria Santos: 1.50
Juan Dela Cruz: 2.30
Ana Reyes: 1.20
Carlo Mendoza: 2.52
Liza Villanueva: 1.32
Pedro Cruz: 2.68
Sofia Ramos: 1.77
Miguel Torres: 2.36
Grace Lim: 1.11
Andrei Bautista: 2.61
Lithium Alkali: 2.34

===== DATABASE FEATURES ====
1. Display Average Grade
2. Display Highest Grade
3. Display Lowest Grade
4. Exit Features Menu
5. Exit Features And main menu
Enter choice: 2

--- Highest Grades for Each Student ---
Maria Santos: PEHM2 - 1.00
Juan Dela Cruz: PEHM2 - 1.75
Ana Reyes: Chem1, Computer Science2, Math2, Social Science2 - 1.00
Carlo Mendoza: PEHM2 - 2.00
Liza Villanueva: Computer Scien